# COLAB SETUP

In [1]:
!git clone https://github.com/GeorgeVasile04/llm_lie_detection_black_vs_white_box.git

Cloning into 'llm_lie_detection_black_vs_white_box'...
remote: Enumerating objects: 676, done.
remote: Counting objects: 100% (273/273), done.
remote: Compressing objects: 100% (203/203), done.
remote: Total 676 (delta 167), reused 168 (delta 70), pack-reused 403 (from 1)
Receiving objects: 100% (676/676), 47.50 MiB | 13.13 MiB/s, done.
Resolving deltas: 100% (257/257), done.
Updating files: 100% (390/390), done.


In [2]:
# Run this cell first if you are executing the notebook in Google Colab.
# It handles module dependencies and connects to your Google Drive to load your project.

try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab. Installing dependencies...")

    # 1. Install required packages
    get_ipython().system('pip install -q transformers accelerate datasets scikit-learn pandas')

    # 2. Mount Google Drive (if using Drive)
    from google.colab import drive
    drive.mount('/content/drive')

    # 3. Setup Project Path - More robust path detection
    import os, sys

    # First, try to find the project root by looking for the main folder
    possible_paths = [
        '/content/drive/MyDrive/llm_lie_detection_black_vs_white_box/Black_vs_White_Lie_Detection/Aligned_Comparison_BB_WB',
        '/content/llm_lie_detection_black_vs_white_box/Black_vs_White_Lie_Detection/Aligned_Comparison_BB_WB',
        os.path.join(os.getcwd(), 'Black_vs_White_Lie_Detection', 'Aligned_Comparison_BB_WB'),
        os.getcwd()  # If already in the right directory
    ]

    base_proj_path = None
    for path in possible_paths:
        if os.path.exists(path) and os.path.exists(os.path.join(path, 'load_data.py')):
            base_proj_path = path
            break

    if base_proj_path:
        os.chdir(base_proj_path)
        print(f"✅ Changed working directory to: {base_proj_path}")
    else:
        print("⚠️ Could not find project directory automatically. Please check your file structure.")
        print(f"Current directory: {os.getcwd()}")

    # Add current directory and parent directories to Python path
    current_dir = os.getcwd()
    if current_dir not in sys.path:
        sys.path.insert(0, current_dir)

    parent_dir = os.path.dirname(current_dir)
    if parent_dir not in sys.path:
        sys.path.insert(0, parent_dir)

    grandparent_dir = os.path.dirname(parent_dir)
    if grandparent_dir not in sys.path:
        sys.path.insert(0, grandparent_dir)

    print(f"✅ Python path updated. Current sys.path includes: {sys.path[:3]}...")

    # 4. Hugging Face Authentication (required for Llama models)
    print("\n🔐 IMPORTANT: Llama models require Hugging Face authentication.")
    print("If you haven't already:")
    print("1. Go to https://huggingface.co/meta-llama/Llama-2-7b-chat-hf")
    print("2. Accept the license agreement")
    print("3. Create a Hugging Face token at https://huggingface.co/settings/tokens")
    print("4. Add it to Colab Secrets (the 🔑 icon on the left) with the name 'HF_TOKEN'")
    print()

    try:
        from huggingface_hub import login
        from google.colab import userdata

        # Load token securely from Colab Secrets
        hf_token = userdata.get('HF_TOKEN')

        if hf_token:
            login(token=hf_token)
            print("✅ Successfully authenticated with Hugging Face!")
        else:
            print("⚠️ Secret 'HF_TOKEN' not found. Falling back to manual login...")
            login()
    except Exception as e:
        print(f"⚠️ Hugging Face login failed: {e}")
        print("Make sure you added the secret named 'HF_TOKEN' in the Colab sidebar.")

except ImportError:
    IN_COLAB = False
    print("Running locally. Setup skipped.")

Running in Google Colab. Installing dependencies...
Mounted at /content/drive
✅ Changed working directory to: /content/llm_lie_detection_black_vs_white_box/Black_vs_White_Lie_Detection/Aligned_Comparison_BB_WB
✅ Python path updated. Current sys.path includes: ['/content/llm_lie_detection_black_vs_white_box', '/content/llm_lie_detection_black_vs_white_box/Black_vs_White_Lie_Detection', '/content/llm_lie_detection_black_vs_white_box/Black_vs_White_Lie_Detection/Aligned_Comparison_BB_WB']...

🔐 IMPORTANT: Llama models require Hugging Face authentication.
If you haven't already:
1. Go to https://huggingface.co/meta-llama/Llama-2-7b-chat-hf
2. Accept the license agreement
3. Create a Hugging Face token at https://huggingface.co/settings/tokens
4. Add it to Colab Secrets (the 🔑 icon on the left) with the name 'HF_TOKEN'

✅ Successfully authenticated with Hugging Face!


# Imports and settings

In [ ]:
import torch
import numpy as np
import pandas as pd
import time
import os
import pickle
from tqdm.auto import tqdm
from load_data import load_dataset_aligned
from wb_activations import load_model, get_activations_for_dataset
from bb_logprobs import compute_bb_features_for_dataset, compute_l1_yes_no_answers
from wb_probes import train_wb_probes
from bb_classifier import train_bb_classifier
from evaluation_utils import compare_results
from Black_vs_White_Lie_Detection.Class_Balance_Impact_BB_WB.wb_probes_imbalance import select_best_layer_cv

# Settings
MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

Using device: cuda


# 1. Load Dataset

In [ ]:
# Define ALL 17 datasets
DATASETS = [
    "commonsense_qa", "race", "arc_easy", "arc_challenge",
    "open_book_qa", "got_cities", "got_sp_en_trans", "got_larger_than",
    "got_cities_cities_conj", "got_cities_cities_disj", "imdb",
    "amazon_polarity", "ag_news", "dbpedia_14", "rte", "copa", "boolq"
]

# Datasets with specific hard-limits
EXACT_SIZE_DATASETS = [
    "commonsense_qa", "race", "arc_easy", "arc_challenge", 
    "imdb", "open_book_qa", "boolq", "rte", "amazon_polarity", 
    "ag_news", "dbpedia_14"
]

WB_BATCH_SIZE = 64
BB_BATCH_SIZE = 50

dataset_dict = {}
print(f"Loading {len(DATASETS)} datasets...\n")

start_time = time.time()
for ds_name in tqdm(DATASETS, desc="Loading datasets"):
    # Determine sizes based on whether it needs exact limits
    if ds_name in EXACT_SIZE_DATASETS:
        max_train = 5000
        max_val = 1000
    else:
        # We will split 5/6 Train and 1/6 Validation inside load_dataset_aligned if left dynamic
        max_train = 5000
        max_val = 1000 
    
    # Load balanced
    ds_train_raw = load_dataset_aligned(ds_name, split='train', n_samples=max_train)
    ds_val_raw = load_dataset_aligned(ds_name, split='validation', n_samples=max_val)

    df_train = pd.DataFrame(ds_train_raw)
    df_val = pd.DataFrame(ds_val_raw)
    
    df = pd.concat([df_train, df_val], ignore_index=True)

    if len(df_train) > 0 and len(df_val) > 0:
        dataset_dict[ds_name] = df
        print(f"Loaded {len(df):,} samples for {ds_name} (train={len(df_train):,}, validation={len(df_val):,})")
    else:
        print(f"Skipped {ds_name}: missing data.")

elapsed = time.time() - start_time
print(f"\n✅ Step 1 Complete: Loaded {len(dataset_dict)} datasets")

# 2. Load Model

In [ ]:

print("Step 2: Loading model...")
import time  # Ensure time is available
start_time = time.time()
model, tokenizer = load_model(MODEL_NAME, device=DEVICE)
elapsed = time.time() - start_time
print(f"✅ Step 2 Complete: Model loaded in {elapsed:.1f}s")

# 3. White Box: Extract Activations

In [ ]:
wb_data_dict = {}

print("Step 3: Extracting White Box Activations...")
start_time = time.time()
outer_pbar = tqdm(dataset_dict.keys(), desc="Overall PB: Extracting WB activations")
for ds_name in outer_pbar:
    # Dynamically lower batch size for memory-heavy datasets like IMDB
    current_batch_size = 28 if ds_name == "imdb" else WB_BATCH_SIZE

    df = dataset_dict[ds_name]
    df_train = df[df["split"] == "train"].reset_index(drop=True)
    df_validation = df[df["split"] == "validation"].reset_index(drop=True)
    outer_pbar.set_description(f"WB Activations: {ds_name} (train/val)")
    wb_train = get_activations_for_dataset(df_train, model, tokenizer, device=DEVICE, batch_size=current_batch_size, show_progress=True)
    wb_validation = get_activations_for_dataset(df_validation, model, tokenizer, device=DEVICE, batch_size=current_batch_size, show_progress=True)
    wb_data_dict[ds_name] = {
        "train": wb_train,
        "validation": wb_validation,
    }

elapsed = time.time() - start_time
print(f"\n✅ Step 3 Complete: White Box activations extracted for {len(wb_data_dict)} datasets in {elapsed:.1f}s")

# 4. Black Box: Compute Logprobs

In [ ]:
# Load the exact elicitation questions used in the original Black Box paper
# The paper selected 48 probes organized into 3 groups: lie, no_lie, and knowable
import numpy as np

try:
    # Load the original paper's probe indices from the Black Box project
    no_lie_indices = np.load("../../Black_Box_Lie_Detection/results/probes_groups/no_lie_indices.npy")
    lie_indices = np.load("../../Black_Box_Lie_Detection/results/probes_groups/lie_indices.npy")
    knowable_indices = np.load("../../Black_Box_Lie_Detection/results/probes_groups/knowable_indices.npy")

    # Concatenate to get the union (48 probes total)
    PAPER_PROBE_INDICES = np.concatenate([no_lie_indices, lie_indices, knowable_indices])
    print(f"✅ Loaded {len(PAPER_PROBE_INDICES)} elicitation questions (exact same as original paper)")
    print(f"   Probe categories: {len(no_lie_indices)} no_lie + {len(lie_indices)} lie + {len(knowable_indices)} knowable")

except FileNotFoundError as e:
    print(f"⚠️ Warning: Could not load original paper's probe indices: {e}")
    print("Will use all available probes instead.")
    PAPER_PROBE_INDICES = None

In [ ]:
l2_data_dict = {}

os.makedirs("bb_elicitation_vectors", exist_ok=True)
DATA_EFF_PATH = os.path.join("..", "Data_Efficiency_BB_WB", "outputs")

print("Step 4a: Computing/Loading Level 2 (Logprobs) Features...")
start_time = time.time()

datasets_list = list(dataset_dict.keys())
with tqdm(total=len(datasets_list), desc="Step 4a — L2 Logprobs", unit="dataset") as pbar:
    for ds_name in datasets_list:
        df = dataset_dict[ds_name]
        df_train = df[df["split"] == "train"].reset_index(drop=True)
        df_validation = df[df["split"] == "validation"].reset_index(drop=True)

        data_eff_path = os.path.join(DATA_EFF_PATH, f"{ds_name}_bb_features_efficiency.pkl")
        local_l2_path = os.path.join("bb_elicitation_vectors", f"{ds_name}_bb_features_lvl2.pkl")

        if os.path.exists(data_eff_path):
            pbar.set_postfix(dataset=ds_name, source="Exp2 cache")
            with open(data_eff_path, "rb") as f:
                cached = pickle.load(f)
            X_train_l2, y_train_l2 = cached["train_full"]
            X_validation_l2, y_validation_l2 = cached["validation"]

        elif os.path.exists(local_l2_path):
            pbar.set_postfix(dataset=ds_name, source="local cache")
            with open(local_l2_path, "rb") as f:
                cached = pickle.load(f)
            X_train_l2, y_train_l2 = cached["train"]
            X_validation_l2, y_validation_l2 = cached["validation"]

        else:
            pbar.set_postfix(dataset=ds_name, source="computing...")
            bb_train = compute_bb_features_for_dataset(
                df_train, model, tokenizer, device=DEVICE,
                batch_size=BB_BATCH_SIZE, probe_indices=PAPER_PROBE_INDICES, show_progress=True
            )
            bb_val = compute_bb_features_for_dataset(
                df_validation, model, tokenizer, device=DEVICE,
                batch_size=BB_BATCH_SIZE, probe_indices=PAPER_PROBE_INDICES, show_progress=True
            )

            X_train_list, y_train_list = [], []
            for res in bb_train:
                if not res.get("bb_features"):
                    continue
                X_train_list.append([p["score"] for p in res["bb_features"]])
                y_train_list.append(res["label"])

            X_val_list, y_val_list = [], []
            for res in bb_val:
                if not res.get("bb_features"):
                    continue
                X_val_list.append([p["score"] for p in res["bb_features"]])
                y_val_list.append(res["label"])

            X_train_l2, y_train_l2 = np.array(X_train_list), np.array(y_train_list)
            X_validation_l2, y_validation_l2 = np.array(X_val_list), np.array(y_val_list)

            with open(local_l2_path, "wb") as f:
                pickle.dump(
                    {"train": (X_train_l2, y_train_l2), "validation": (X_validation_l2, y_validation_l2)}, f
                )

        l2_data_dict[ds_name] = {
            "X_train": X_train_l2, "y_train": y_train_l2,
            "X_validation": X_validation_l2, "y_validation": y_validation_l2,
        }
        pbar.update(1)

elapsed = time.time() - start_time
print(f"\n✅ Step 4a Complete: L2 features ready for {len(l2_data_dict)} datasets in {elapsed:.1f}s")


In [ ]:
l1_data_dict = {}

os.makedirs("bb_elicitation_vectors_lvl_1", exist_ok=True)

print("Step 4b: Computing/Loading Level 1 (Yes/No) Features...")
start_time = time.time()

datasets_list = list(dataset_dict.keys())
with tqdm(total=len(datasets_list), desc="Step 4b — L1 Yes/No", unit="dataset") as outer_pbar:
    for ds_name in datasets_list:
        df = dataset_dict[ds_name]
        df_train = df[df["split"] == "train"].reset_index(drop=True)
        df_validation = df[df["split"] == "validation"].reset_index(drop=True)

        local_l1_path = os.path.join("bb_elicitation_vectors_lvl_1", f"{ds_name}_bb_features_lvl1.pkl")

        if os.path.exists(local_l1_path):
            outer_pbar.set_postfix(dataset=ds_name, source="local cache")
            with open(local_l1_path, "rb") as f:
                cached = pickle.load(f)
            X_train_l1, y_train_l1 = cached["train"]
            X_validation_l1, y_validation_l1 = cached["validation"]

        else:
            outer_pbar.set_postfix(dataset=ds_name, source="computing...")

            l1_train = compute_l1_yes_no_answers(
                df_train, model, tokenizer, device=DEVICE,
                batch_size=BB_BATCH_SIZE, probe_indices=PAPER_PROBE_INDICES,
                show_progress=True, desc=f"{ds_name} [train] — {len(df_train)} samples"
            )

            l1_val = compute_l1_yes_no_answers(
                df_validation, model, tokenizer, device=DEVICE,
                batch_size=BB_BATCH_SIZE, probe_indices=PAPER_PROBE_INDICES,
                show_progress=True, desc=f"{ds_name} [val]   — {len(df_validation)} samples"
            )

            X_train_l1 = np.array([res["l1_features"] for res in l1_train])
            y_train_l1 = np.array([res["label"] for res in l1_train])
            X_validation_l1 = np.array([res["l1_features"] for res in l1_val])
            y_validation_l1 = np.array([res["label"] for res in l1_val])

            with open(local_l1_path, "wb") as f:
                pickle.dump({"train": (X_train_l1, y_train_l1), "validation": (X_validation_l1, y_validation_l1)}, f)

            print(f"  ✅ {ds_name} — L1 extraction succeeded (train={len(X_train_l1)}, val={len(X_validation_l1)})")

        l1_data_dict[ds_name] = {
            "X_train": X_train_l1, "y_train": y_train_l1,
            "X_validation": X_validation_l1, "y_validation": y_validation_l1,
        }
        outer_pbar.update(1)

elapsed = time.time() - start_time
print(f"✅ Step 4b Complete: L1 features ready for {len(l1_data_dict)} datasets in {elapsed:.1f}s")


# 5. Train & Evaluate Classifiers

In [ ]:
print("Step 5: Training Classifiers (L1, L2, L3)")
all_comparisons = []
LAYER_LIST = list(range(1, 33, 2))
print(f"Testing the following layers: {LAYER_LIST}")

wb_methods = ["lr", "dim", "lat", "ccs", "lda", "pca", "pca-g", "lr-g"]
method_names_map = {
    "lr": "Logistic Regression",
    "dim": "Difference in Means",
    "lat": "Linear Artificial Tomography",
    "ccs": "Contrast-Consistent Search",
    "lda": "Linear Discriminant Analysis",
    "pca": "Principal Component Analysis",
    "pca-g": "Grouped PCA",
    "lr-g": "Grouped Logistic Regression",
}

In [ ]:
import pickle
import os

l1_results = []
datasets_list = list(l1_data_dict.keys())
with tqdm(total=len(datasets_list), desc="Step 5 — L1 Classifiers", unit="dataset") as pbar:
    for ds_name in datasets_list:
        pbar.set_postfix(dataset=ds_name)
        X_train, y_train = l1_data_dict[ds_name]["X_train"], l1_data_dict[ds_name]["y_train"]
        X_test, y_test = l1_data_dict[ds_name]["X_validation"], l1_data_dict[ds_name]["y_validation"]

        bb_model, bb_metric = train_bb_classifier(X_train, X_test, y_train, y_test)
        bb_metric.update({"Dataset": ds_name, "Access_Level": "Level 1 (Yes/No)"})
        l1_results.append(bb_metric)
        pbar.update(1)

pd.DataFrame(l1_results).to_csv("level1_results.csv", index=False)
print("✅ Saved Level 1 results to level1_results.csv")


In [ ]:
os.makedirs("classifiers/level2", exist_ok=True)

l2_results = []
datasets_list = list(l2_data_dict.keys())
with tqdm(total=len(datasets_list), desc="Step 5 — L2 Classifiers", unit="dataset") as pbar:
    for ds_name in datasets_list:
        pbar.set_postfix(dataset=ds_name)
        X_train, y_train = l2_data_dict[ds_name]["X_train"], l2_data_dict[ds_name]["y_train"]
        X_test, y_test = l2_data_dict[ds_name]["X_validation"], l2_data_dict[ds_name]["y_validation"]

        bb_bundle, bb_metric = train_bb_classifier(X_train, X_test, y_train, y_test)
        bb_metric.update({"Dataset": ds_name, "Access_Level": "Level 2 (Logprobs)"})
        l2_results.append(bb_metric)

        # Save clf + scaler together so Experiment 4 can apply the source scaler to target datasets
        with open(f"classifiers/level2/{ds_name}_l2_classifier.pkl", "wb") as f:
            pickle.dump(bb_bundle, f)
        pbar.update(1)

pd.DataFrame(l2_results).to_csv("level2_results.csv", index=False)
print("✅ Saved Level 2 results to level2_results.csv")


In [ ]:
os.makedirs("classifiers/level3", exist_ok=True)

l3_results = []
datasets_list = list(wb_data_dict.keys())
with tqdm(total=len(datasets_list), desc="Step 5 — L3 Classifiers", unit="dataset") as outer_pbar:
    for ds_name in datasets_list:
        wb_train = wb_data_dict[ds_name]["train"]
        wb_validation = wb_data_dict[ds_name]["validation"]

        with tqdm(total=len(wb_methods), desc=f"  {ds_name}", unit="method", leave=False) as method_pbar:
            for method in wb_methods:
                method_pbar.set_postfix(method=method, stage="CV layer selection")
                best_layer = select_best_layer_cv(
                    wb_train, layer_list=LAYER_LIST, method=method, n_splits=4, verbose=False
                )

                method_pbar.set_postfix(method=method, stage=f"training L{best_layer}")
                trained_probes, wb_metrics = train_wb_probes(
                    wb_train, eval_data=wb_validation, layer_list=[best_layer], method=method
                )

                metrics = wb_metrics[best_layer]
                metrics.update({
                    "Dataset": ds_name, "Access_Level": "Level 3",
                    "Method": method_names_map[method], "Layer": best_layer,
                })
                l3_results.append(metrics)

                best_probe = trained_probes[best_layer]
                with open(f"classifiers/level3/{ds_name}_{method}_l3_classifier.pkl", "wb") as f:
                    pickle.dump(best_probe, f)
                method_pbar.update(1)

        outer_pbar.set_postfix(dataset=ds_name)
        outer_pbar.update(1)

pd.DataFrame(l3_results).to_csv("level3_results.csv", index=False)
print("\n✅ Saved Level 3 results to level3_results.csv")


In [ ]:
# Save WB activations for all datasets (LAYER_LIST layers only) so Experiment 4
# can evaluate L3 probes on target datasets without reloading the model.
import gc

os.makedirs("wb_activations", exist_ok=True)

print("Saving WB activations for E4 (LAYER_LIST layers only)...")
datasets_list = list(wb_data_dict.keys())
with tqdm(total=len(datasets_list), desc="Saving WB activations", unit="dataset") as pbar:
    for ds_name in datasets_list:
        save_path = os.path.join("wb_activations", f"{ds_name}_wb_activations.pkl")

        if os.path.exists(save_path):
            pbar.set_postfix(dataset=ds_name, status="already saved")
            pbar.update(1)
            continue

        pbar.set_postfix(dataset=ds_name, status="saving...")

        def extract_layer_arrays(split_data):
            X = {layer: np.array([item["activations"][layer] for item in split_data]) for layer in LAYER_LIST}
            y = np.array([item["label"] for item in split_data])
            return {"X": X, "y": y}

        payload = {
            "train":      extract_layer_arrays(wb_data_dict[ds_name]["train"]),
            "validation": extract_layer_arrays(wb_data_dict[ds_name]["validation"]),
        }

        with open(save_path, "wb") as f:
            pickle.dump(payload, f)

        del payload
        gc.collect()
        pbar.update(1)

print("\n✅ WB activations saved to wb_activations/")
print("   Format per file: {'train': {'X': {layer: array(n,4096)}, 'y': array(n,)}, 'validation': {...}}")
print("   E4 load pattern:")
print("     data = pickle.load(open('wb_activations/got_cities_wb_activations.pkl','rb'))")
print("     X_target = data['validation']['X'][best_layer]  # shape: (n_val, 4096)")
print("     y_target = data['validation']['y']")


# 6. Comparison

In [ ]:
# Step 6: Summary and Comparison
print("Step 6: Summary and Comparison per Access Level")

df_l1 = pd.DataFrame(l1_results)
df_l2 = pd.DataFrame(l2_results)
df_l3 = pd.DataFrame(l3_results)

# 1. Base Summary
print("\n--- Level 1 Summary ---")
print(df_l1.groupby("Dataset")[["AUC", "MAP", "BRP_90"]].mean().sort_values("AUC", ascending=False))

print("\n--- Level 2 Summary ---")
print(df_l2.groupby("Dataset")[["AUC", "MAP", "BRP_90"]].mean().sort_values("AUC", ascending=False))

print("\n--- Level 3 Summary (Best probe per method) ---")
l3_avg_per_dataset = df_l3.groupby("Dataset")[["AUC", "MAP", "BRP_90"]].mean().sort_values("AUC", ascending=False)
print("Average performance of WB methods per dataset:")
print(l3_avg_per_dataset)

print("\n--- Level 3 Method Ranking (Average across datasets) ---")
l3_avg_per_method = df_l3.groupby("Method")[["AUC", "MAP", "BRP_90"]].mean().sort_values("AUC", ascending=False)
print(l3_avg_per_method)

# Compare overall Access Level Performances
print("\n🏆 COMPACT COMPARISON: BLACK BOX VS BEST WHITE BOX 🏆")
compact_rows = []
for ds_name in df_l1["Dataset"].unique():
    l1_row = df_l1[df_l1["Dataset"] == ds_name]
    l2_row = df_l2[df_l2["Dataset"] == ds_name]
    l3_dataset = df_l3[df_l3["Dataset"] == ds_name]
    
    # Best whitebox probe
    best_wb = l3_dataset.loc[l3_dataset["AUC"].idxmax()] if not l3_dataset.empty else None
    
    compact_rows.append({
        "Dataset": ds_name,
        "L1 (Yes/No) AUC": f"{l1_row['AUC'].values[0]:.3f}" if not l1_row.empty else "N/A",
        "L2 (Logprobs) AUC": f"{l2_row['AUC'].values[0]:.3f}" if not l2_row.empty else "N/A",
        "Best L3 WB AUC": f"{best_wb['AUC']:.3f} ({best_wb['Method']}, L{best_wb['Layer']})" if best_wb is not None else "N/A",
    })

compact_df = pd.DataFrame(compact_rows)

from IPython.display import display
display(compact_df)

compact_df.to_csv("compact_comparison_results.csv", index=False)
print("\n✅ Saved Compact format to: compact_comparison_results.csv")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualizing the Performance ceiling under access constraints (RQ1)
print("\n📊 Generating Visualizations for RQ1...")

# Prepare data for plotting
plot_data = []

for ds_name in df_l1["Dataset"].unique():
    l1_val = df_l1[df_l1["Dataset"] == ds_name]["AUC"].values
    l2_val = df_l2[df_l2["Dataset"] == ds_name]["AUC"].values
    
    # Get the best White Box AUC for this dataset
    l3_ds = df_l3[df_l3["Dataset"] == ds_name]
    best_l3_val = l3_ds["AUC"].max() if not l3_ds.empty else None
    
    if len(l1_val) > 0: plot_data.append({"Dataset": ds_name, "Access Level": "Level 1 (Yes/No)", "AUC": l1_val[0]})
    if len(l2_val) > 0: plot_data.append({"Dataset": ds_name, "Access Level": "Level 2 (Logprobs)", "AUC": l2_val[0]})
    if best_l3_val is not None: plot_data.append({"Dataset": ds_name, "Access Level": "Level 3 (White Box)", "AUC": best_l3_val})

plot_df = pd.DataFrame(plot_data)

# 1. Bar Plot of Average Performance per Access Level
plt.figure(figsize=(8, 6))
sns.barplot(data=plot_df, x="Access Level", y="AUC", errorbar="sd", palette="viridis")
plt.title("Average Lie Detection Performance by Access Level (RQ1)", fontsize=14, pad=15)
plt.ylabel("AUC Score", fontsize=12)
plt.xlabel("Model Access Level", fontsize=12)
plt.ylim(0.4, 1.0)
plt.axhline(0.5, color='red', linestyle='--', alpha=0.5, label="Random Guess (0.5)")
plt.legend()
plt.tight_layout()
plt.savefig("access_level_average_performance.png", dpi=300)
plt.show()

# 2. Detailed Dataset-by-Dataset Comparison
plt.figure(figsize=(14, 8))
sns.barplot(data=plot_df, x="Dataset", y="AUC", hue="Access Level", palette="viridis")
plt.title("Lie Detection Performance across Datasets (L1 vs L2 vs L3)", fontsize=14, pad=15)
plt.ylabel("AUC Score", fontsize=12)
plt.xlabel("Dataset", fontsize=12)
plt.ylim(0.4, 1.0)
plt.axhline(0.5, color='red', linestyle='--', alpha=0.5)
plt.xticks(rotation=45, ha="right")
plt.legend(title="Access Level", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("access_level_dataset_breakdown.png", dpi=300)
plt.show()